# Training Data Preparation

Loads Discord conversation data from HuggingFace, takes the first N examples, and saves them in SFT format for fine-tuning.

No train/test split needed — evaluation is done separately on NYCC/Jester.

## NYCC Benchmark Notes

**Quality Ranking task construction (from the paper):**

- **Winner (A):** Official finalists OR top-5 crowd-ranked captions (semantically deduplicated to 3 unique via SBERT)
- **Distractor (B):** Middle tertile of crowd ratings — top and bottom thirds discarded. For older contests without ratings, bottom 25% (typos, formatting issues) filtered by ML model.
- **Matching:** Pairs are matched on word count, character count, and punctuation count. SBERT ensures the two captions are semantically dissimilar (same punchline not allowed).

**Implication:** This is *finalist vs. average human*, not *winner vs. obvious loser*. Surface-level features (length, punctuation) are controlled for — the model must understand humour content to do well.

**Human accuracy baselines (from §3.1):**
- Editor selections (NYAcc): 64.6%
- Crowd selections (CrowdAcc): 83.7%

**Our models (Qwen3 base, validation folds 0–3):** ~54–61% — below human on both targets.

**Evaluation target:** Use `winner_source == 'crowd_winner'` pairs (statistically more reliable; more annotators).

## Config

In [ ]:
HF_DATASET_NAME = "mookiezi/Discord-Dialogues"
N = 50_000
OUTPUT_PATH = "../datasets/training/discord_sft.jsonl"

## 1. Login & Load

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from datasets import load_dataset

ds = load_dataset(HF_DATASET_NAME, split="train")
print(ds)
print(ds.features)

In [ ]:
# Inspect a few examples
for ex in ds.select(range(3)):
    print(ex["text"])
    print("---")

## 2. Take First N

In [ ]:
subset = ds.select(range(N))
print(f"Subset size: {len(subset)}")

## 3. Parse ChatML → Message List

The dataset stores conversations as a single `text` string in ChatML format.
We parse it into a list of `{role, content}` dicts — the standard format expected by unsloth/TRL.

In [ ]:
import re

TURN_RE = re.compile(r"<\|im_start\|>(\w+)\n(.*?)<\|im_end\|>", re.DOTALL)

def parse_chatml(text: str) -> list[dict]:
    return [
        {"role": role, "content": content.strip()}
        for role, content in TURN_RE.findall(text)
    ]

# Sanity check
sample = parse_chatml(subset[0]["text"])
for turn in sample:
    print(turn)

## 4. Save as JSONL

In [ ]:
import json
from pathlib import Path

out = Path(OUTPUT_PATH)
out.parent.mkdir(parents=True, exist_ok=True)

skipped = 0
with open(out, "w") as f:
    for ex in subset:
        messages = parse_chatml(ex["text"])
        if len(messages) < 2:
            skipped += 1
            continue
        f.write(json.dumps({"messages": messages}) + "\n")

print(f"Saved {N - skipped} examples to {out} ({skipped} skipped — failed to parse)")